# 🎮 WAKA Tournament — ตรวจสอบการชำระเงิน

**วิธีใช้:**
1. รันทีละ Cell จากบนลงล่าง (กด ▶ หรือ Shift+Enter)
2. Cell ที่ 3 — แก้ไข Sheet IDs และค่าสมัครให้ตรง
3. Cell ที่ 5 — อัปโหลดไฟล์ CSV หรือ PDF จากธนาคาร
4. Cell สุดท้าย — รันแล้วเปิด Google Sheet ดูผล

> ⚠️ **ชื่อบัญชีที่โอน**: กรอก**ภาษาไทย**ตามบัญชีธนาคาร เช่น `รัญญศักดิ์ แจ้งในเมือง` (ชื่อ English จะ match ไม่ได้)

In [ ]:
# Cell 1: ติดตั้ง packages (รันครั้งเดียวต่อ session)
!pip install -q gspread google-auth pdfplumber

In [ ]:
# Cell 2: ล็อกอิน Google (จะขึ้นหน้าต่างให้อนุญาต)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
print('✅ ล็อกอิน Google สำเร็จ')

In [ ]:
# Cell 3: ⚙️ ตั้งค่า — แก้ไขตรงนี้ก่อนรัน

OUTPUT_SHEET_ID = '1WSfd9sKHl2H5O7Ai1DvqVaL7Tuwni-nfBc-hTX8HilA'

EVENTS = [
    {
        'name': 'Lorcana',
        'source_sheet_id': '1vEzBdnQ1doPH3KLADSN9XbrC5tViZiGq1kLWET-6LeQ',
        'gid': 118500649,
        'fee': 350,
    },
    {
        'name': 'Riftbound',
        'source_sheet_id': '1EIqD91sJGBpwN0AeU9w8ior67elMIQS1vyY2mwShLZU',
        'gid': 935532850,
        'fee': [500, 900],  # 2 ราคา — match ได้ทั้งคู่
    },
    {
        'name': 'Pokemon Champions',
        'source_sheet_id': '1HLZ_tM3WbCiPP6BtuHSKmBo2H5KkYizsh71lJ4_moJA',
        'gid': 2026372639,
        'fee': 200,
    },
    {
        'name': 'Pokemon TCG',
        'source_sheet_id': '1eHQQ-8ANDJfMcUyVwtrtumfgrtnTaiZr--wzUFExMpE',
        'gid': 759768958,
        'fee': 250,
    },
    {
        'name': 'BOT',
        'source_sheet_id': '1K7AFCxOOPzw-kp0kzUf07d-96dCsKnHjkYbIemzxeDM',
        'gid': 2054056416,
        'fee': 250,
    },
]

print(f'✅ ตั้งค่าแล้ว: {len(EVENTS)} event(s)')

In [ ]:
# Cell 4: โหลด functions ทั้งหมด (ไม่ต้องแก้ไข)
import csv, io, json, re
from pathlib import Path

# ── Keywords ──────────────────────────────────────────────
CSV_COLUMN_KEYWORDS = {
    'txn_id': ['เลขที่รายการ', 'transaction', 'ref no', 'refno'],
    'date':   ['วันทำรายการ', 'date', 'วันที่'],
    'sender': ['รับเงินจาก', 'sender', 'จากบัญชี', 'from'],
    'amount': ['จำนวนเงิน', 'amount', 'ยอด'],
    'bank':   ['source of fund', 'source', 'ธนาคารต้นทาง'],
}

FORM_COLUMN_KEYWORDS = {
    'game_name':     ['in game name', 'ingame', 'แข่งในวงการ', 'ชื่อแข่ง', 'trainer id', 'openchat'],
    'openchat_name': ['openchat', 'open chat'],
    'facebook':      ['facebook', 'เฟสบุค', 'fb'],
    'transfer_name': ['ชื่อบัญชี', 'ชื่อที่โอน', 'ชื่อโอน', 'ใช้โอน', 'ใช้ในการโอน',
                      'transfer name', 'ชื่อเจ้าของบัญชี', 'ชื่อที่ใช้'],
    'slip_url':      ['สลิป', 'slip', 'หลักฐาน', 'การชำระ', 'payment', 'อัพโหลด'],
}

OUTPUT_HEADER = [
    '#', 'ชื่อที่ใช้แข่ง', 'ชื่อใน OpenChat', 'ชื่อเฟสบุค', 'ชื่อบัญชีที่โอน',
    'สถานะ', 'รายละเอียด', 'ยอดที่พบ', 'ชื่อในธนาคาร', 'เลขที่รายการ', 'วันที่โอน', 'ลิงค์สลิป',
]

# ── Helpers ───────────────────────────────────────────────
def normalize(name):
    return re.sub(r'[\s.\-_]', '', name).lower()

def detect_columns(header, keywords):
    cols = {}
    for col_key, kws in keywords.items():
        for i, h in enumerate(header):
            if any(kw.lower() in h.lower() for kw in kws):
                cols[col_key] = i
                break
    return cols

def load_bank_csv(content_bytes):
    for encoding in ('utf-8-sig', 'utf-8', 'cp874', 'tis-620'):
        try:
            text = content_bytes.decode(encoding)
            try:
                dialect = csv.Sniffer().sniff(text[:2000], delimiters='|,\t')
            except csv.Error:
                dialect = csv.excel
            rows = list(csv.reader(io.StringIO(text), dialect))
            result = _parse_csv_rows(rows)
            if result:
                return result
        except UnicodeDecodeError:
            continue
    raise ValueError('ไม่สามารถอ่าน CSV ได้ — ลอง save as UTF-8')

def load_bank_pdf(content_bytes):
    import pdfplumber
    transactions = []
    txn_re = re.compile(
        r'(T30-\d{4}-\d{4}-\d{4}-\d+)'
        r'\s+(.+?)'
        r'\s+([\d,]+\.\d{2})'
        r'\s+([A-Z]{2,8})',
        re.UNICODE
    )
    date_re = re.compile(r'(\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}:\d{2})')
    with pdfplumber.open(io.BytesIO(content_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            for line in text.split('\n'):
                m = txn_re.search(line)
                if not m:
                    continue
                txn_id = m.group(1)
                sender = m.group(2).strip()
                amount_str = m.group(3)
                bank = m.group(4)
                pre = line[:m.start()]
                dm = date_re.search(pre)
                date = dm.group(1) if dm else ''
                try:
                    amount = float(amount_str.replace(',', ''))
                except ValueError:
                    continue
                transactions.append({
                    'txn_id': txn_id,
                    'date': date,
                    'sender': sender,
                    'amount': amount,
                    'bank': bank,
                })
    if not transactions:
        raise ValueError('ไม่พบ transactions ใน PDF — ตรวจสอบว่าเป็น PDF แบบ text (ไม่ใช่รูปภาพ)')
    return transactions

def _parse_csv_rows(rows):
    header_idx, cols = None, {}
    for i, row in enumerate(rows):
        row_lower = [c.lower().strip() for c in row]
        matches = sum(
            1 for kws in CSV_COLUMN_KEYWORDS.values()
            if any(kw in cell for kw in kws for cell in row_lower)
        )
        if matches >= 2:
            header_idx = i
            for col_key, kws in CSV_COLUMN_KEYWORDS.items():
                for j, cell in enumerate(row_lower):
                    if any(kw in cell for kw in kws):
                        cols[col_key] = j
                        break
            break
    if header_idx is None:
        raise ValueError('หา header ใน CSV ไม่เจอ — ตรวจสอบว่ามีคอลัมน์ เลขที่รายการ, รับเงินจาก, จำนวนเงิน')

    transactions = []
    for row in rows[header_idx + 1:]:
        if not row or not any(row):
            continue
        def cell(k, _row=row):
            i = cols.get(k)
            return _row[i].strip() if i is not None and i < len(_row) else ''
        try:
            amount = float(cell('amount').replace(',', ''))
        except ValueError:
            continue
        t = {'txn_id': cell('txn_id'), 'date': cell('date'),
             'sender': cell('sender'), 'amount': amount, 'bank': cell('bank')}
        if t['sender'] or t['txn_id']:
            transactions.append(t)
    return transactions

def find_matching_txn(transfer_name, facebook_name, expected_fees, bank_rows, used_txn_ids):
    if not isinstance(expected_fees, list):
        expected_fees = [float(expected_fees)]
    match_name = transfer_name if transfer_name else facebook_name
    n_match = normalize(match_name)
    best_amount = None
    for txn in bank_rows:
        if txn['txn_id'] and txn['txn_id'] in used_txn_ids:
            continue
        amount_ok = any(abs(txn['amount'] - f) < 1.0 for f in expected_fees)
        n_sender = normalize(txn['sender'])
        name_ok = bool(n_match and n_sender and (n_match in n_sender or n_sender in n_match))
        if name_ok and amount_ok:
            return txn, '✅', f"ยืนยันแล้ว | โอน {txn['amount']:.0f}฿ | {txn['sender']}"
        if amount_ok and best_amount is None:
            best_amount = txn
    if best_amount:
        t = best_amount
        label = f'บัญชี: {match_name}'
        return t, '⚠️', f"ตรวจสอบชื่อ | ยอดตรง {t['amount']:.0f}฿ | ธนาคาร: {t['sender']} | {label}"
    fees_str = '/'.join(f'{f:.0f}' for f in expected_fees)
    return None, '❌', f'ไม่พบในรายงานธนาคาร | บัญชี: {match_name} | คาดยอด {fees_str}฿'

print('✅ โหลด functions เรียบร้อย')

In [ ]:
# Cell 5: 📂 อัปโหลดไฟล์ CSV หรือ PDF จากธนาคาร
from google.colab import files

print('กด "Choose Files" แล้วเลือกไฟล์ CSV หรือ PDF จากธนาคาร')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
content_bytes = uploaded[filename]

if filename.lower().endswith('.pdf') or content_bytes[:4] == b'%PDF':
    bank_rows = load_bank_pdf(content_bytes)
else:
    bank_rows = load_bank_csv(content_bytes)

print(f'✅ โหลด {len(bank_rows)} transactions จาก {filename}')

In [ ]:
# Cell 6: 🚀 รันตรวจสอบและเขียนผลลัพธ์ไป Google Sheet

used_txn_ids = set()
output_sheet = gc.open_by_key(OUTPUT_SHEET_ID)
grand_total = 0

def name_similarity(form_name, bank_name):
    """คะแนนความคล้ายชื่อ 0.0–1.0 (substring = 1.0)"""
    n_f = normalize(form_name)
    n_b = normalize(bank_name)
    if not n_f or not n_b:
        return 0.0
    if n_f in n_b or n_b in n_f:
        return 1.0
    common = sum(1 for c in n_f if c in n_b)
    return common / max(len(n_f), len(n_b))

for event in EVENTS:
    name = event['name']
    expected_fees = event['fee'] if isinstance(event['fee'], list) else [float(event['fee'])]
    print(f"\n{'='*50}")
    print(f"Event: {name}  |  ค่าสมัคร: {'/'.join(f'{f:.0f}' for f in expected_fees)}฿")

    src_sh = gc.open_by_key(event['source_sheet_id'])
    src_ws = src_sh.get_worksheet_by_id(event['gid']) if 'gid' in event else src_sh.sheet1
    rows = src_ws.get_all_values()

    if len(rows) < 2:
        print('  ไม่มีข้อมูล — ข้าม')
        continue

    header, data_rows = rows[0], rows[1:]
    cols = detect_columns(header, FORM_COLUMN_KEYWORDS)

    try:
        ws = output_sheet.worksheet(name)
        ws.clear()
    except gspread.WorksheetNotFound:
        ws = output_sheet.add_worksheet(title=name, rows=500, cols=len(OUTPUT_HEADER))
    ws.append_row(OUTPUT_HEADER)

    # --- Parse all form rows ---
    parsed = []
    for seq, row in enumerate(data_rows, start=1):
        def cell(key, _row=row):
            i = cols.get(key)
            return _row[i].strip() if i is not None and i < len(_row) else ''
        game_name     = cell('game_name') or cell('openchat_name')
        openchat_name = cell('openchat_name')
        facebook      = cell('facebook')
        transfer_name = cell('transfer_name')
        slip_url      = cell('slip_url')
        if game_name:
            parsed.append((seq, game_name, openchat_name, facebook, transfer_name, slip_url))

    # --- Pass 1: lock ✅ (ชื่อ + ยอดตรงทั้งคู่) ก่อนใครเลย ---
    exact_matches = {}
    txn_to_owner  = {}  # txn_id → game_name ของคนที่ได้ transaction นี้
    for seq, game_name, openchat_name, facebook, transfer_name, slip_url in parsed:
        txn, status, detail = find_matching_txn(
            transfer_name, facebook, expected_fees, bank_rows, used_txn_ids
        )
        if status == '✅' and txn:
            used_txn_ids.add(txn['txn_id'])
            txn_to_owner[txn['txn_id']] = game_name
            exact_matches[seq] = (txn, status, detail)

    # --- Pass 2: ⚠️ — จับทุก (row ที่ยังไม่ match) × (txn ที่ยังไม่ถูกใช้)
    #            เรียงจากคล้ายชื่อที่สุดก่อน แล้ว greedy assign ---
    candidates = []  # (similarity, seq, game_name, match_name, txn)
    for seq, game_name, openchat_name, facebook, transfer_name, slip_url in parsed:
        if seq in exact_matches:
            continue
        match_name = transfer_name if transfer_name else facebook
        for txn in bank_rows:
            if txn['txn_id'] and txn['txn_id'] in used_txn_ids:
                continue
            if any(abs(txn['amount'] - f) < 1.0 for f in expected_fees):
                sim = name_similarity(match_name, txn['sender'])
                candidates.append((sim, seq, game_name, match_name, txn))

    candidates.sort(key=lambda x: -x[0])
    warn_matches = {}
    for sim, seq, gname, match_name, txn in candidates:
        if seq in warn_matches:
            continue
        if txn['txn_id'] and txn['txn_id'] in used_txn_ids:
            continue
        detail = f"ตรวจสอบชื่อ | ยอดตรง {txn['amount']:.0f}฿ | ธนาคาร: {txn['sender']} | บัญชี: {match_name}"
        warn_matches[seq] = (txn, '⚠️', detail)
        used_txn_ids.add(txn['txn_id'])
        txn_to_owner[txn['txn_id']] = gname

    # --- Compile results ---
    results = []
    for seq, game_name, openchat_name, facebook, transfer_name, slip_url in parsed:
        if seq in exact_matches:
            txn, status, detail = exact_matches[seq]
        elif seq in warn_matches:
            txn, status, detail = warn_matches[seq]
        else:
            match_name = transfer_name if transfer_name else facebook
            fees_str   = '/'.join(f'{f:.0f}' for f in expected_fees)

            # 🚫 สลิปซ้ำ — ตรวจเฉพาะตอนที่ชื่อตรงจริงๆ (substring match = 1.0)
            # ป้องกัน false positive จากคนชื่อคล้ายกันแต่คนละคน
            best_txn, best_sim = None, 0.0
            for t in bank_rows:
                if any(abs(t['amount'] - f) < 1.0 for f in expected_fees):
                    sim = name_similarity(match_name, t['sender'])
                    if sim > best_sim:
                        best_sim, best_txn = sim, t

            if best_txn and best_sim == 1.0 and best_txn['txn_id'] in txn_to_owner:
                owner  = txn_to_owner[best_txn['txn_id']]
                txn    = best_txn
                status = '🚫'
                detail = f"สลิปซ้ำ | ใช้ไปแล้วโดย: {owner} | T30: {best_txn['txn_id']}"
            else:
                txn    = None
                status = '❌'
                detail = f'ไม่พบในรายงานธนาคาร | บัญชี: {match_name} | คาดยอด {fees_str}฿'

        txn_id   = txn['txn_id'] if txn else ''
        txn_date = txn['date']   if txn else ''
        txn_amt  = f"{txn['amount']:.0f}" if txn else ''
        txn_name = txn['sender'] if txn else ''

        print(f'  [{seq:02d}] {game_name:<20} {status} {detail[:60]}')
        results.append([seq, game_name, openchat_name, facebook, transfer_name,
                        status, detail, txn_amt, txn_name, txn_id, txn_date, slip_url])

    if results:
        ws.append_rows(results)

    confirmed = sum(1 for r in results if r[5] == '✅')
    warned    = sum(1 for r in results if r[5] == '⚠️')
    duped     = sum(1 for r in results if r[5] == '🚫')
    failed    = len(results) - confirmed - warned - duped
    print(f'\n  ผล: {confirmed} ✅  {warned} ⚠️  {duped} 🚫  {failed} ❌  (จาก {len(results)} รายการ)')
    grand_total += len(results)

print(f"\n{'='*50}")
print(f'เสร็จสิ้น! ประมวลผลทั้งหมด {grand_total} รายการ')
print(f'ดูผล: https://docs.google.com/spreadsheets/d/{OUTPUT_SHEET_ID}')